# Install requested Libraries and Import

In [1]:
!pip install -q langgraph
!pip install -q langchain
!pip install -q langchain-groq
!pip install -q plotly
!pip install -q openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00


In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from dotenv import load_dotenv
from langchain_groq import ChatGroq
warnings.filterwarnings("ignore")

# Groq API

In [ ]:
load_dotenv()
os.environ["GROQ_API_KEY"] = ""
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# Initialize LLM
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# Test AI

In [4]:
response = llm.invoke(
    "Explain Data Analysis in one paragraph."
)
print(response.content)

Data analysis is the process of evaluating and interpreting data to extract meaningful insights and patterns. It involves a series of steps, including data collection, cleaning, and transformation, followed by the application of statistical and analytical techniques to identify trends, correlations, and relationships within the data. Data analysis can be used to answer specific questions, solve problems, or inform business decisions, and it often involves the use of tools and technologies such as spreadsheets, statistical software, and data visualization platforms. By applying data analysis techniques, individuals and organizations can gain a deeper understanding of their data, identify areas for improvement, and make informed decisions to drive business outcomes, optimize operations, and improve overall performance. Effective data analysis requires a combination of technical skills, business acumen, and critical thinking, and it is a key component of many fields, including business, h

# File Loader

In [5]:
def load_data(path):
    extension = path.split(".")[-1].lower()
    if extension == "csv":
        return pd.read_csv(path)
    elif extension in ["xlsx", "xls"]:
        return pd.read_excel(path)
    elif extension == "json":
        return pd.read_json(path)
    else:
        raise Exception("Unsupported File")

# Test Loader
df = load_data("/content/sales_data.csv")
df.head()

,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
2,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,South-David
3,1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
4,1061,2023-03-24,Charlie,East,3750.20,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie


In [6]:
# Data Profiling Function
def profile_dataset(df):
  profile = {}
  # Basic Information
  profile["Rows"] = df.shape[0]
  profile["Columns"] = df.shape[1]
  # Column Names
  profile["Column Names"] = list(df.columns)
  # Data Types
  profile["Data Types"] = df.dtypes.astype(str).to_dict()
  # Missing Values
  profile["Missing Values"] = df.isnull().sum().to_dict()
  # Duplicate Rows
  profile["Duplicate Rows"] = int(df.duplicated().sum())
  # Memory Usage
  profile["Memory Usage (MB)"] = round(
      df.memory_usage(deep=True).sum() / 1024**2,2)
  # Numeric Columns
  profile["Numeric Columns"] = list(
      df.select_dtypes(include=np.number).columns)
  # Categorical Columns
  profile["Categorical Columns"] = list(
      df.select_dtypes(include=["object","category"]).columns)
  # Datetime Columns
  profile["Datetime Columns"] = list(
      df.select_dtypes(include=["datetime64"]).columns)
  return profile

# Run Profiling
profile = profile_dataset(df)

# Display Profile
for key, value in profile.items():
    print("="*60)
    print(key)
    print(value)

Rows
1000
Columns
14
Column Names
['Product_ID', 'Sale_Date', 'Sales_Rep', 'Region', 'Sales_Amount', 'Quantity_Sold', 'Product_Category', 'Unit_Cost', 'Unit_Price', 'Customer_Type', 'Discount', 'Payment_Method', 'Sales_Channel', 'Region_and_Sales_Rep']
Data Types
{'Product_ID': 'int64', 'Sale_Date': 'object', 'Sales_Rep': 'object', 'Region': 'object', 'Sales_Amount': 'float64', 'Quantity_Sold': 'int64', 'Product_Category': 'object', 'Unit_Cost': 'float64', 'Unit_Price': 'float64', 'Customer_Type': 'object', 'Discount': 'float64', 'Payment_Method': 'object', 'Sales_Channel': 'object', 'Region_and_Sales_Rep': 'object'}
Missing Values
{'Product_ID': 0, 'Sale_Date': 0, 'Sales_Rep': 0, 'Region': 0, 'Sales_Amount': 0, 'Quantity_Sold': 0, 'Product_Category': 0, 'Unit_Cost': 0, 'Unit_Price': 0, 'Customer_Type': 0, 'Discount': 0, 'Payment_Method': 0, 'Sales_Channel': 0, 'Region_and_Sales_Rep': 0}
Duplicate Rows
0
Memory Usage (MB)
0.48
Numeric Columns
['Product_ID', 'Sales_Amount', 'Quantity_So

In [7]:
# Statistical Summary
def statistical_summary(df):
    numeric = df.select_dtypes(include=np.number)
    if numeric.empty:
        return "No Numeric Columns Found."
    return numeric.describe().T
statistical_summary(df)

# Missiing Value Report
def missing_report(df):
    report = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Percentage":
        (df.isnull().sum()/len(df))*100})
    report = report.sort_values(
        by="Percentage",
        ascending=False)
    return report
missing_report(df)

# Duplicate Report
duplicate = df[df.duplicated()]
duplicate

# Unique Values
def unique_values(df):
    data = pd.DataFrame({
        "Unique Values":
        df.nunique()})
    return data
unique_values(df)

# Data Quality Score
def data_quality_score(df):
    score = 100
    missing = (df.isnull().sum().sum()/(df.shape[0]*df.shape[1]))*100
    duplicate = (df.duplicated().sum()/len(df))*100
    score = score - missing - duplicate
    return round(max(score,0),2)
score = data_quality_score(df)
print("Data Quality Score:",score,"%")

# Smart Dataset Detector
def detect_dataset_type(df):
    columns = [c.lower() for c in df.columns]
    if any("sales" in c for c in columns):
        return "Sales Dataset"
    elif any("loan" in c for c in columns):
        return "Finance Dataset"
    elif any("employee" in c for c in columns):
        return "HR Dataset"
    elif any("ph" in c for c in columns):
        return "Water Quality Dataset"
    else:
        return "General Dataset"
dataset_type = detect_dataset_type(df)
print(dataset_type)

Data Quality Score: 100.0 %
Sales Dataset


In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
# Create LLM model
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0)
# Graph State
class AgentState(TypedDict):
    dataframe: object
    profile: dict
    insight: str

# Profiling Agent
def profiling_agent(state):
    df = state["dataframe"]
    profile = profile_dataset(df)
    state["profile"] = profile
    return state

# Insight Agent
def insight_agent(state):
    profile = state["profile"]
    prompt = f"""
You are a Senior Data Analyst.
Analyze this dataset profile.
{profile}
Generate:
1. Dataset Summary
2. Data Quality
3. Possible Business Insights
4. Possible Problems
5. Recommended Analysis
Answer in professional format.
"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state

# Build LangGraph
graph = StateGraph(AgentState)
graph.add_node("profiling",profiling_agent)
graph.add_node("insight",insight_agent)
graph.set_entry_point("profiling")
graph.add_edge("profiling","insight")
graph.add_edge(
    "insight",
    END)
workflow = graph.compile()

# Run Agent
result = workflow.invoke(

    {
        "dataframe": df,
        "profile": {},
        "insight": ""})

# AI Response
print(result["insight"])

**Dataset Analysis Report**

### 1. Dataset Summary

The provided dataset contains 1000 rows and 14 columns, with a mix of numeric and categorical data types. The dataset appears to be related to sales data, with columns such as `Sales_Amount`, `Quantity_Sold`, `Unit_Cost`, and `Unit_Price` providing information on sales transactions. The dataset also includes columns related to sales representatives, regions, product categories, and customer types, which could be useful for analyzing sales performance and trends.

**Key Statistics:**

* Number of rows: 1000
* Number of columns: 14
* Numeric columns: 6
* Categorical columns: 8
* Missing values: 0
* Duplicate rows: 0
* Memory usage: 0.48 MB

### 2. Data Quality

The dataset appears to be of high quality, with no missing values and no duplicate rows. The data types for each column are consistent, with numeric columns containing numeric data and categorical columns containing string data. However, the `Sale_Date` column is currently store

In [9]:
# Save Profile
profile = result["profile"]
profile
# Save AI Insight
insight = result["insight"]
print(insight)

**Dataset Analysis Report**

### 1. Dataset Summary

The provided dataset contains 1000 rows and 14 columns, with a mix of numeric and categorical data types. The dataset appears to be related to sales data, with columns such as `Sales_Amount`, `Quantity_Sold`, `Unit_Cost`, and `Unit_Price` providing information on sales transactions. The dataset also includes columns related to sales representatives, regions, product categories, and customer types, which could be useful for analyzing sales performance and trends.

**Key Statistics:**

* Number of rows: 1000
* Number of columns: 14
* Numeric columns: 6
* Categorical columns: 8
* Missing values: 0
* Duplicate rows: 0
* Memory usage: 0.48 MB

### 2. Data Quality

The dataset appears to be of high quality, with no missing values and no duplicate rows. The data types for each column are consistent, with numeric columns containing numeric data and categorical columns containing string data. However, the `Sale_Date` column is currently store

In [10]:
# Detect Column Types
def detect_columns(df):
    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns.tolist()
    categorical_cols = df.select_dtypes(
        include=["object","category"]
    ).columns.tolist()
    datetime_cols = df.select_dtypes(
        include=["datetime64[ns]"]
    ).columns.tolist()
    return {
        "numeric": numeric_cols,
        "categorical": categorical_cols,
        "datetime": datetime_cols
    }
columns = detect_columns(df)
columns

# Histogram Tool
def histogram_tool(df,column):
    fig = px.histogram(df,x=column,title=f"Distribution of {column}")
    fig.show()

# Bar Chart Tool
def bar_chart_tool(df,cat,num):
    grouped = df.groupby(cat)[num].sum().reset_index()
    fig = px.bar(grouped,x=cat,y=num,title=f"{num} by {cat}")
    fig.show()

# Scatter Plot Tool
def scatter_tool(df,x,y):
    fig = px.scatter(df,x=x,y=y,title=f"{x} vs {y}")
    fig.show()

# Correlation Heatmap
def heatmap_tool(df):
    numeric = df.select_dtypes(include=np.number)
    corr = numeric.corr()
    fig = px.imshow(corr,text_auto=True,title="Correlation Matrix")
    fig.show()

# Line Chart
def line_chart_tool(df,date_col,num_col):
    fig = px.line(df,x=date_col,y=num_col,title=f"{num_col} Trend")
    fig.show()

# Smart Visualization Agent
def visualization_agent(state):
    df = state["dataframe"]
    cols = detect_columns(df)
    numeric = cols["numeric"]
    categorical = cols["categorical"]
    datetime = cols["datetime"]
    charts = []

    # Histogram

    if len(numeric) >= 1:
        histogram_tool(df,numeric[0])
        charts.append(f"Histogram ({numeric[0]})")

    # Bar Chart

    if len(categorical)>=1 and len(numeric)>=1:
        bar_chart_tool(df,categorical[0],numeric[0])
        charts.append("Bar Chart")

    # Scatter Plot

    if len(numeric)>=2:
        scatter_tool(df,numeric[0],numeric[1])
        charts.append("Scatter Plot")

    # Heatmap

    if len(numeric)>=2:
        heatmap_tool(df)
        charts.append("Correlation Heatmap")

    # Line Chart

    if len(datetime)>=1 and len(numeric)>=1:
      line_chart_tool(df,datetime[0],numeric[0])
      charts.append("Line Chart")
      state["charts"] = charts
      return state

In [11]:
# Update State
# Update previous AgentState
from typing import List

class AgentState(TypedDict):
    dataframe: object
    profile: dict
    insight: str
    charts: List[str]

# Update LangGraph
graph = StateGraph(AgentState)
graph.add_node("profiling",profiling_agent)
graph.add_node("visualization",visualization_agent)
graph.add_node("insight",insight_agent)
graph.set_entry_point("profiling")
graph.add_edge("profiling","visualization")
graph.add_edge("visualization","insight")
graph.add_edge("insight",END)
workflow = graph.compile()

# Run Workflow
result = workflow.invoke({
    "dataframe":df,"profile":{},"charts":[],"insight":""})
# Show Charts Used
print(result["charts"])

# AI Chart Explanation
prompt = f"""
Dataset Profile
{result["profile"]}
Charts Generated
{result["charts"]}
Explain these charts like a senior business analyst."""
response = llm.invoke(prompt)
print(response.content)

[]
As a senior business analyst, I must point out that there are no charts generated for this dataset. However, I can provide an overview of the dataset and suggest potential charts that could be useful for analysis.

The dataset appears to be related to sales data, with 14 columns and 1000 rows. The columns include product information, sales details, customer information, and regional data. The data types are a mix of integers, floats, and objects, indicating a combination of numerical and categorical data.

Some potential charts that could be useful for analysis include:

1. **Sales Amount by Region**: A bar chart or map to visualize the total sales amount by region, which could help identify top-performing regions.
2. **Sales Rep Performance**: A bar chart to compare the sales performance of different sales representatives, which could help identify top-performing reps and areas for improvement.
3. **Product Category Sales**: A pie chart or bar chart to show the sales distribution b

In [12]:
# Define Tools State
from typing import TypedDict, List, Dict, Any
class AgentState(TypedDict):
    dataframe: object
    profile: dict
    charts: List[str]
    insight: str
    tool_decision: Dict[str, Any]

# Tool Registry
TOOLS = {
    "histogram": histogram_tool,
    "bar_chart": bar_chart_tool,
    "scatter": scatter_tool,
    "heatmap": heatmap_tool,
    "line_chart": line_chart_tool}

# Tool Decision Agent
import json
def tool_decision_agent(state):
    profile = state["profile"]
    prompt = f"""
You are an expert Data Analyst.
Dataset Profile:
{profile}
Return ONLY valid JSON.
Format:
{{"charts":[{{"tool":"histogram","column":"Sales"}}]}}
Choose the best charts for this dataset."""
    response = llm.invoke(prompt)
    try:
        decision = json.loads(response.content)
    except:
        decision = {"charts":[]}
    state["tool_decision"] = decision
    return state

# Tools Executor
def tool_executor(state):
    df = state["dataframe"]
    decisions = state["tool_decision"]
    generated = []
    for item in decisions["charts"]:
        tool = item["tool"]
        if tool == "histogram":
            histogram_tool(df,item["column"])
            generated.append(tool)
        elif tool == "heatmap":
            heatmap_tool(df)
            generated.append(tool)
        elif tool == "scatter":
          scatter_tool(df,item["x"],item["y"])
          generated.append(tool)
        elif tool == "bar_chart":
            bar_chart_tool(df,item["category"],item["value"])
            generated.append(tool)
        elif tool == "line_chart":
            line_chart_tool(df,item["date"],item["value"])
            generated.append(tool)
    state["charts"] = generated
    return state

# Improve Insight Agent
def insight_agent(state):
    prompt = f"""
You are a Senior Business Analyst.
Dataset Profile
{state['profile']}
Charts Generated
{state['charts']}
Generate
1. Executive Summary
2. Data Quality
3. Key Insights
4. Business Risks
5. Recommendations
Professional Report"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state
# Build Workflow
graph = StateGraph(AgentState)
graph.add_node("profiling",profiling_agent)
graph.add_node("tool_decision",tool_decision_agent)
graph.add_node("tool_executor",tool_executor)
graph.add_node("insight",insight_agent)
graph.set_entry_point("profiling")
graph.add_edge("profiling","tool_decision")
graph.add_edge("tool_decision","tool_executor")
graph.add_edge("tool_executor","insight")
graph.add_edge("insight",END)
workflow = graph.compile()

# Run Agent
result = workflow.invoke({
    "dataframe": df,
    "profile": {},
    "charts": [],
    "tool_decision": {},
    "insight": ""})

In [13]:
# Output
print(result["tool_decision"])
print(result["charts"])
print(result["insight"])

{'charts': []}
[]
**Professional Report**

### 1. Executive Summary

The provided dataset contains 1000 rows and 14 columns, offering insights into sales data across various regions, sales representatives, and product categories. The data is largely complete, with no missing values and no duplicate rows. This report aims to analyze the dataset, identify key trends and insights, and provide recommendations for business improvement.

### 2. Data Quality

The dataset exhibits high data quality, with:

* **No missing values**: All columns have zero missing values, indicating complete data.
* **No duplicate rows**: The absence of duplicate rows suggests that each record is unique.
* **Appropriate data types**: The data types assigned to each column (e.g., int64, object, float64) appear to be suitable for the respective data.
* **Low memory usage**: The dataset requires minimal memory (0.48 MB), making it efficient for analysis.

However, the 'Sale_Date' column is currently stored as an obje

In [14]:
from abc import ABC, abstractmethod
class BaseTool(ABC):
    @property
    @abstractmethod
    def name(self):
        pass
    @property
    @abstractmethod
    def description(self):
        pass
    @abstractmethod
    def execute(self, df, params):
        pass
# Histogram Tool Class
class HistogramTool(BaseTool):
    @property
    def name(self):
        return "histogram"
    @property
    def description(self):
        return "Create histogram for numeric column"
    def execute(self, df, params):
        column = params["column"]
        fig = px.histogram(df,x=column,title=f"Distribution of {column}")
        fig.show()
        return f"Histogram Created: {column}"
# Heatmap Tool Class
class HeatmapTool(BaseTool):
    @property
    def name(self):
        return "heatmap"
    @property
    def description(self):
        return "Correlation heatmap"
    def execute(self, df, params):
        corr = df.select_dtypes(include=np.number).corr()
        fig = px.imshow(corr,text_auto=True)
        fig.show()
        return "Heatmap Created"
# Bar Chart Tool Class
class BarChartTool(BaseTool):
    @property
    def name(self):
        return "bar_chart"
    @property
    def description(self):
        return "Category vs Numeric Analysis"
    def execute(self, df, params):
        cat = params["category"]
        num = params["value"]
        grouped = df.groupby(cat)[num].sum().reset_index()
        fig = px.bar(grouped,x=cat,y=num)
        fig.show()
        return "Bar Chart Created"
# Professional Tool Registry
TOOL_REGISTRY = {
    "histogram": HistogramTool(),
    "heatmap": HeatmapTool(),
    "bar_chart": BarChartTool()}
# View Available Tools
for tool_name, tool in TOOL_REGISTRY.items():
    print(tool_name)
    print(tool.description)
    print("-"*50)

histogram
Create histogram for numeric column
--------------------------------------------------
heatmap
Correlation heatmap
--------------------------------------------------
bar_chart
Category vs Numeric Analysis
--------------------------------------------------


In [15]:
# Better Tool Decisiom Agent
import json
import pandas as pd
import numpy as np
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Data Profiling Function (Copied from PFj4R4ErJD36)
def profile_dataset(df):
  profile = {}
  profile["Rows"] = df.shape[0]
  profile["Columns"] = df.shape[1]
  profile["Column Names"] = list(df.columns)
  profile["Data Types"] = df.dtypes.astype(str).to_dict()
  profile["Missing Values"] = df.isnull().sum().to_dict()
  profile["Duplicate Rows"] = int(df.duplicated().sum())
  profile["Memory Usage (MB)"] = round(
      df.memory_usage(deep=True).sum() / 1024**2,2)
  profile["Numeric Columns"] = list(
      df.select_dtypes(include=np.number).columns)
  profile["Categorical Columns"] = list(
      df.select_dtypes(include=["object","category"]).columns)
  profile["Datetime Columns"] = list(
      df.select_dtypes(include=["datetime64"]).columns)
  return profile

# Profiling Agent (Copied from iTg6P2VyLoOX)
def profiling_agent(state):
    df = state["dataframe"]
    profile = profile_dataset(df)
    state["profile"] = profile
    return state

# Insight Agent (Copied from cb0yCchykx38)
def insight_agent(state):
    prompt = f"""
You are a Senior Business Analyst.
Dataset Profile
{state['profile']}
Charts Generated
{state['charts']}
Generate
1. Executive Summary
2. Data Quality
3. Key Insights
4. Business Risks
5. Recommendations
Professional Report"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state

def tool_decision_agent(state):
    profile = state["profile"]
    available_tools = []
    for name, tool in TOOL_REGISTRY.items():
        available_tools.append({"name": tool.name,"description": tool.description})
    prompt = f"""
You are an expert Data Analyst.
Dataset Profile:
{profile}
Available Tools:
{available_tools}
Return valid JSON only.
Example:
{{"tools":[{{"name":"histogram","params": {{"column":"Sales"}}}}]}}"""
    response = llm.invoke(prompt)
    try:
        decision = json.loads(response.content)
    except:
        decision = {"tools":[]}
    state["tool_decision"] = decision
    return state
# Safe Tool Executor
def tool_executor(state):
    df = state["dataframe"]
    decisions = state["tool_decision"]
    results = []
    for tool_info in decisions["tools"]:
        try:
            tool_name = tool_info["name"]
            params = tool_info["params"]
            tool = TOOL_REGISTRY[
                tool_name]
            output = tool.execute(df,params)
            results.append(output)
        except Exception as e:
            results.append(f"Error: {str(e)}")
    state["charts"] = results
    return state

# Update State

class AgentState(TypedDict):
    dataframe: object
    profile: dict
    tool_decision: dict
    charts: list
    insight: str

# Rebuild Graph
graph = StateGraph(AgentState)
graph.add_node("profiling",profiling_agent)
graph.add_node("tool_decision",tool_decision_agent)
graph.add_node("tool_executor",tool_executor)
graph.add_node("insight",insight_agent)
graph.set_entry_point("profiling")
graph.add_edge("profiling","tool_decision")
graph.add_edge("tool_decision","tool_executor")
graph.add_edge("tool_executor","insight")
graph.add_edge("insight",END)
workflow = graph.compile()

# Test
result = workflow.invoke({
    "dataframe": df,"profile": {},"tool_decision": {},"charts": [],"insight": ""})
# Results
print(result["tool_decision"])
print(result["charts"])
print(result["insight"])

{'tools': []}
[]
**Professional Report**

### 1. Executive Summary

The provided dataset contains 1000 rows and 14 columns, offering insights into sales transactions. The data includes information about products, sales representatives, regions, sales amounts, and customer types. With no missing values and no duplicate rows, the dataset is clean and ready for analysis. This report aims to provide an overview of the data quality, key insights, potential business risks, and recommendations for future improvements.

### 2. Data Quality

The dataset exhibits high data quality, with the following notable characteristics:
- **No Missing Values**: All 1000 rows have complete information across all 14 columns, indicating thorough data collection or effective data cleaning.
- **No Duplicate Rows**: The absence of duplicate rows suggests that each sales transaction is unique, which is crucial for accurate analysis and decision-making.
- **Data Types**: The mix of numeric and categorical columns a

In [ ]:
import json
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

# Create LLM model (re-initializing here for clarity, though it's already defined)
llm = ChatGroq(
    groq_api_key="", # Placeholder, use actual key or os.getenv
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# Update AgentState
class AgentState(TypedDict):
    dataframe: object
    profile: dict
    charts: List[str]
    insight: str
    tool_decision: Dict[str, Any]
    errors: List[str]
    user_question: str
    answer: str

# Dataset Summary Tool
def dataset_summary_tool(df):
    summary = {
        "rows": len(df),
        "columns": list(df.columns),
        "numeric_columns":
            df.select_dtypes(include="number").columns.tolist(),
        "categorical_columns":
            df.select_dtypes(include=["object","category"]).columns.tolist()}
    return summary

# Statistic Tool
def statistics_tool(df):
    return df.describe(include="all")

# Top Rows Tool
def preview_tool(df, n=5):
  return df.head(n)

# Note: `register_tool` calls are removed as they are not defined and replaced by `TOOL_REGISTRY` in other cells.

# Profiling Agent (copied from iTg6P2VyLoOX for self-containment)
def profile_dataset(df):
  profile = {}
  profile["Rows"] = df.shape[0]
  profile["Columns"] = df.shape[1]
  profile["Column Names"] = list(df.columns)
  profile["Data Types"] = df.dtypes.astype(str).to_dict()
  profile["Missing Values"] = df.isnull().sum().to_dict()
  profile["Duplicate Rows"] = int(df.duplicated().sum())
  profile["Memory Usage (MB)"] = round(
      df.memory_usage(deep=True).sum() / 1024**2,2)
  profile["Numeric Columns"] = list(
      df.select_dtypes(include=np.number).columns)
  profile["Categorical Columns"] = list(
      df.select_dtypes(include=["object","category"]).columns)
  profile["Datetime Columns"] = list(
      df.select_dtypes(include=["datetime64"]).columns)
  return profile

def profiling_agent(state):
    df = state["dataframe"]
    profile = profile_dataset(df)
    state["profile"] = profile
    return state

# Insight Agent (copied from cb0yCchykx38 for self-containment)
def insight_agent(state):
    prompt = f"""
You are a Senior Business Analyst.
Dataset Profile
{state['profile']}
Charts Generated
{state['charts']}
Generate
1. Executive Summary
2. Data Quality
3. Key Insights
4. Business Risks
5. Recommendations
Professional Report"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state

# Chat Agent
def chat_agent(state):
    df = state["dataframe"]
    question = state["user_question"]
    summary = dataset_summary_tool(df)
    preview = preview_tool(df)
    stats = statistics_tool(df)
    prompt = f"""
You are an expert Data Analyst.
Dataset Summary
{summary}
Preview
{preview}
Statistics
{stats}
User Question
{question}
Rules:
1. Answer only using the provided dataset information.
2. If information is unavailable, clearly say so.
3. Give a concise, professional answer.
"""
    response = llm.invoke(prompt)
    state["answer"] = response.content
    return state

# Add Chart Node
graph = StateGraph(AgentState)

graph.add_node("profiling", profiling_agent)
# Removed 'planner' and 'executor' as they are not defined in the current context
# If you intend to use them, please ensure they are properly defined.
graph.add_node("insight", insight_agent)
graph.add_node("chat", chat_agent)
graph.set_entry_point("profiling")
graph.add_edge("profiling", "insight") # Connect profiling to insight directly
graph.add_edge("insight", "chat")
graph.add_edge("chat", END)
workflow = graph.compile()

In [17]:
# Ask a Question
question = "Which numeric columns are present in the dataset?"
result = workflow.invoke({
    "dataframe": df,
    "profile": {},
    "charts": [],
    "tool_decision": {},
    "insight": "",
    "errors": [],
    "user_question": question,
    "answer": ""})

# Print Answer
print(result["answer"])

The numeric columns present in the dataset are:

1. Product_ID
2. Sales_Amount
3. Quantity_Sold
4. Unit_Cost
5. Unit_Price
6. Discount


In [18]:
# Update State
class AgentState(TypedDict):
    dataframe: object
    profile: dict
    charts: List[str]
    insight: str
    tool_decision: Dict[str, Any]
    errors: List[str]
    user_question: str
    query_plan: Dict[str, Any]
    query_result: Any
    answer: str

# Query Planer
def query_planner_agent(state):
    profile = state["profile"]
    question = state["user_question"]
    prompt = f"""
You are an AI Data Analyst.
Dataset Profile:
{profile}
User Question:
{question}
Return ONLY JSON.
Supported operations:
count
sum
mean
max
min
groupby_sum
groupby_mean
sort
top_n
Example:
{{
"operation":"groupby_sum",
"group_column":"Product",
"value_column":"Sales"
}}
"""
    response = llm.invoke(prompt)
    try:
        state["query_plan"] = json.loads(response.content)
    except:
        state["query_plan"] = {}
    return state

# Pandas Execution Engine
def pandas_query_engine(state):
    df = state["dataframe"]
    plan = state["query_plan"]
    operation = plan.get("operation")
    result = None
    try:
        if operation == "count":
            result = len(df)
        elif operation == "sum":
            result = df[
                plan["column"]
            ].sum()
        elif operation == "mean":
            result = df[
                plan["column"]
            ].mean()
        elif operation == "max":
            result = df[
                plan["column"]
            ].max()
        elif operation == "min":
            result = df[
                plan["column"]
            ].min()
        elif operation == "groupby_sum":
            result = (
                df.groupby(
                    plan["group_column"]
                )[
                    plan["value_column"]
                ].sum().sort_values(ascending=False))
        elif operation == "groupby_mean":
            result = (df.groupby(
                    plan["group_column"])[plan["value_column"]].mean())
        elif operation == "sort":
            result = (df.sort_values(by=plan["column"],ascending=False))
        elif operation == "top_n":
            result = (df.nlargest(plan["n"],plan["column"]))
    except Exception as e:
        result = str(e)
    state["query_result"] = result
    return state

# Explanation Agent
def answer_agent(state):
    prompt = f"""
You are a Senior Data Analyst.
User Question
{state["user_question"]}
Actual Query Result
{state["query_result"]}
Explain the result professionally.
Do not modify the numbers.
Only explain.
"""
    response = llm.invoke(prompt)
    state["answer"] = response.content
    return state

# Build Workflow
graph = StateGraph(AgentState)
graph.add_node("query_planner",query_planner_agent)
graph.add_node("query_engine",pandas_query_engine)
graph.add_node("answer",answer_agent)
graph.set_entry_point("query_planner")
graph.add_edge("query_planner","query_engine")
graph.add_edge("query_engine","answer")
graph.add_edge("answer",END)
chat_workflow = graph.compile()

In [19]:
# Example
result = chat_workflow.invoke({
    "dataframe":df,
    "profile":profile,
    "user_question":
    "Show total sales by product.",
    "query_plan":{},
    "query_result":"",
    "answer":""})

# View Query Plan
print(result["query_plan"])
{
'operation':'groupby_sum',
'group_column':'Product',
'value_column':'Sales'}

# Actual Pandas Result
print(result["query_result"])

# AI Explanation
print(result["answer"])

{}
None
The query result indicates that there is no data available to display the total sales by product. This suggests that either the data is empty, the query is incorrect, or there is an issue with the data retrieval process. As a Senior Data Analyst, I would investigate further to determine the cause of this result, checking the data sources, query syntax, and database connections to ensure that the query is properly executed and that the data is accurately retrieved.


In [20]:
# Update state
class AgentState(TypedDict):
    dataframe: object
    user_question: str
    profile: dict
    cleaned: bool
    charts: List[str]
    query_plan: Dict[str, Any]
    query_result: Any
    insight: str
    answer: str
    route: List[str]
    errors: List[str]

# Supervisor Agent
def supervisor_agent(state):
    profile = state["profile"]
    question = state["user_question"]
    prompt = f"""
You are the Supervisor AI Agent.
Dataset Profile
{profile}
User Question
{question}
Available Agents
1 Profiling
2 Cleaning
3 Visualization
4 Query
5 Insight
Return ONLY JSON.
Example
{{
"route":[
"profiling",
"visualization",
"insight"
]
}}
"""
    response = llm.invoke(prompt)
    try:
        state["route"] = json.loads(
            response.content
        )["route"]
    except:
        state["route"] = ["profiling","insight"]
    return state

# Claning Agent
def cleaning_agent(state):
    df = state["dataframe"].copy()
    df = df.drop_duplicates()
    for column in df.columns:
        if df[column].dtype == "object":
            mode = df[column].mode()
            if len(mode):
                df[column] = df[column].fillna(mode[0])
        else:
            df[column] = df[column].fillna(
                df[column].median())
    state["dataframe"] = df
    state["cleaned"] = True
    return state

# Query Agent
def query_agent(state):
    state = query_planner_agent(state)
    state = pandas_query_engine(state)
    return state

# Visualization Agent
def visualization_agent(state):
    state = tool_planner_agent(state)
    state = robust_tool_executor(state)
    return state

# Business Insight Agent
def business_insight_agent(state):
    prompt = f"""
You are a Senior Business Analyst.
Dataset Profile
{state["profile"]}
Query Result
{state["query_result"]}
Charts
{state["charts"]}
Generate
Executive Summary
Key Insights
Business Risks
Recommendations
Next Steps
"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state

# Dynamic Router
def router(state):
    return state["route"][0]

# Build LangGrapgraph = StateGraph(AgentState)
graph.add_node("supervisor",supervisor_agent)
graph.add_node("profiling",profiling_agent)
graph.add_node("cleaning",cleaning_agent)
graph.add_node("visualization",visualization_agent)
graph.add_node("query",query_agent)
graph.add_node("insight",business_insight_agent)
graph.set_entry_point("supervisor")



In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from abc import ABC, abstractmethod

# Create LLM model (re-initializing here for clarity, though it's already defined)
llm = ChatGroq(
    groq_api_key="", # Placeholder, use actual key or os.getenv
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# Agent State (Unified)
class AgentState(TypedDict):
    dataframe: object
    profile: dict
    charts: List[str]
    query_plan: Dict[str, Any]
    query_result: Any
    insight: str
    answer: str
    errors: List[str]
    memory: List[str]
    user_question: str
    tool_decision: Dict[str, Any] # Added for tool planning
    cleaned: bool # Added for cleaning agent
    route: List[str] # Added for supervisor routing

# Data Profiling Function (Copied from _nAoHYsx1hnJ)
def profile_dataset(df):
  profile = {}
  profile["Rows"] = df.shape[0]
  profile["Columns"] = df.shape[1]
  profile["Column Names"] = list(df.columns)
  profile["Data Types"] = df.dtypes.astype(str).to_dict()
  profile["Missing Values"] = df.isnull().sum().to_dict()
  profile["Duplicate Rows"] = int(df.duplicated().sum())
  profile["Memory Usage (MB)"] = round(
      df.memory_usage(deep=True).sum() / 1024**2,2)
  profile["Numeric Columns"] = list(
      df.select_dtypes(include=np.number).columns)
  profile["Categorical Columns"] = list(
      df.select_dtypes(include=["object","category"]).columns)
  profile["Datetime Columns"] = list(
      df.select_dtypes(include=["datetime64"]).columns)
  return profile

# Base Tool Class
class BaseTool(ABC):
    @property
    @abstractmethod
    def name(self):
        pass
    @property
    @abstractmethod
    def description(self):
        pass
    @abstractmethod
    def execute(self, df, params):
        pass

# Histogram Tool Class
class HistogramTool(BaseTool):
    @property
    def name(self):
        return "histogram"
    @property
    def description(self):
        return "Create histogram for numeric column"
    def execute(self, df, params):
        column = params["column"]
        fig = px.histogram(df,x=column,title=f"Distribution of {column}")
        fig.show()
        return f"Histogram Created: {column}"

# Heatmap Tool Class
class HeatmapTool(BaseTool):
    @property
    def name(self):
        return "heatmap"
    @property
    def description(self):
        return "Correlation heatmap"
    def execute(self, df, params):
        corr = df.select_dtypes(include=np.number).corr()
        fig = px.imshow(corr,text_auto=True,title="Correlation Matrix")
        fig.show()
        return "Heatmap Created"

# Bar Chart Tool Class
class BarChartTool(BaseTool):
    @property
    def name(self):
        return "bar_chart"
    @property
    def description(self):
        return "Category vs Numeric Analysis"
    def execute(self, df, params):
        cat = params["category"]
        num = params["value"]
        grouped = df.groupby(cat)[num].sum().reset_index()
        fig = px.bar(grouped,x=cat,y=num,title=f"{num} by {cat}")
        fig.show()
        return f"Bar Chart Created: {num} by {cat}"

# Scatter Plot Tool Class
class ScatterTool(BaseTool):
    @property
    def name(self):
        return "scatter"
    @property
    def description(self):
        return "Create a scatter plot for two numeric columns"
    def execute(self, df, params):
        x = params["x"]
        y = params["y"]
        fig = px.scatter(df, x=x, y=y, title=f"{x} vs {y}")
        fig.show()
        return f"Scatter Plot Created: {x} vs {y}"

# Line Chart Tool Class
class LineChartTool(BaseTool):
    @property
    def name(self):
        return "line_chart"
    @property
    def description(self):
        return "Create a line chart for a datetime column and a numeric column"
    def execute(self, df, params):
        date_col = params["date"]
        num_col = params["value"]
        # Ensure the date column is datetime type
        df[date_col] = pd.to_datetime(df[date_col])
        fig = px.line(df, x=date_col, y=num_col, title=f"{num_col} Trend")
        fig.show()
        return f"Line Chart Created: {num_col} Trend over {date_col}"

# Professional Tool Registry (Consolidated)
TOOL_REGISTRY = {
    "histogram": HistogramTool(),
    "heatmap": HeatmapTool(),
    "bar_chart": BarChartTool(),
    "scatter": ScatterTool(),
    "line_chart": LineChartTool()
}

# Profiling Agent (Copied from _nAoHYsx1hnJ)
def profiling_agent(state):
    df = state["dataframe"]
    profile = profile_dataset(df)
    state["profile"] = profile
    return state

# Tool Planner Agent (Adapted from tool_decision_agent in 5P_rgYq1qVjK)
def tool_planner_agent(state):
    profile = state["profile"]
    available_tools = []
    for name, tool in TOOL_REGISTRY.items():
        available_tools.append({"name": tool.name, "description": tool.description})

    user_question = state.get("user_question", "")

    prompt = f"""
You are an expert Data Analyst. Your goal is to select the most appropriate tools to address the user's question based on the dataset profile.
Dataset Profile:
{profile}
Available Tools:
{available_tools}
User Question: {user_question}

Return ONLY valid JSON.
Format:
{{"tools":[{{"name":"histogram","params": {{"column":"Sales"}}}}, {{"name":"bar_chart","params": {{"category":"Region","value":"Sales_Amount"}}}}]}}
Choose the best charts for this dataset based on the profile and user question. If no specific question, choose general exploratory charts.
"""
    response = llm.invoke(prompt)
    try:
        decision = json.loads(response.content)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error in tool_planner_agent: {e}")
        print(f"LLM Response: {response.content}")
        decision = {"tools":[]}
        state["errors"].append(f"JSON parsing error in tool_planner_agent: {e}")
        state["errors"].append(f"LLM output: {response.content}")

    state["tool_decision"] = decision
    return state

# Robust Tool Executor (Adapted from tool_executor in 5P_rgYq1qVjK)
def robust_tool_executor(state):
    df = state["dataframe"]
    decisions = state["tool_decision"]
    generated_charts = []
    errors = state.get("errors", [])

    if not decisions or "tools" not in decisions:
        errors.append("No tool decisions found in state['tool_decision'].")
        state["errors"] = errors
        state["charts"] = generated_charts # Ensure charts list is present
        return state

    for tool_info in decisions["tools"]:
        try:
            tool_name = tool_info["name"]
            params = tool_info.get("params", {}) # Use .get to handle missing params

            if tool_name not in TOOL_REGISTRY:
                errors.append(f"Tool '{tool_name}' not found in TOOL_REGISTRY.")
                continue

            tool = TOOL_REGISTRY[tool_name]
            output = tool.execute(df, params)
            generated_charts.append(output)
        except Exception as e:
            errors.append(f"Error executing tool '{tool_name}' with params {params}: {str(e)}")

    state["charts"] = generated_charts
    state["errors"] = errors
    return state


# Insight Agent (Copied from _nAoHYsx1hnJ)
def insight_agent(state):
    prompt = f"""
You are a Senior Business Analyst.
Dataset Profile
{state['profile']}
Charts Generated
{state['charts']}
Generate
1. Executive Summary
2. Data Quality
3. Key Insights
4. Business Risks
5. Recommendations
Professional Report"""
    response = llm.invoke(prompt)
    state["insight"] = response.content
    return state

# Memory Manager
class MemoryManager:
    def __init__(self):
        self.memory = []
    def add(self, message):
        self.memory.append(message)
    def get(self):
        return self.memory
memory = MemoryManager()

# Save Memory
def save_memory(state):
    memory.add({
        "question": state["user_question"],
        "answer": state["answer"]})
    state["memory"] = memory.get()
    return state

# Memory Agent
def memory_agent(state):
    history = memory.get()
    prompt = f"""
Conversation History
{history}
Current Question
{state["user_question"]}
Answer professionally.
"""
    response = llm.invoke(prompt)
    state["answer"] = response.content
    return state

# Query Planer (Copied from MLyXnLhM7OeM)
def query_planner_agent(state):
    profile = state["profile"]
    question = state["user_question"]
    prompt = f"""
You are an AI Data Analyst.
Dataset Profile:
{profile}
User Question:
{question}
Return ONLY JSON.
Supported operations:
count
sum
mean
max
min
groupby_sum
groupby_mean
sort
top_n
Example:
{{
"operation":"groupby_sum",
"group_column":"Product",
"value_column":"Sales"
}}
"""
    response = llm.invoke(prompt)
    try:
        state["query_plan"] = json.loads(response.content)
    except:
        state["query_plan"] = {}
    return state

# Pandas Execution Engine (Copied from MLyXnLhM7OeM)
def pandas_query_engine(state):
    df = state["dataframe"]
    plan = state["query_plan"]
    operation = plan.get("operation")
    result = None
    try:
        if operation == "count":
            result = len(df)
        elif operation == "sum":
            result = df[
                plan["column"]
            ].sum()
        elif operation == "mean":
            result = df[
                plan["column"]
            ].mean()
        elif operation == "max":
            result = df[
                plan["column"]
            ].max()
        elif operation == "min":
            result = df[
                plan["column"]
            ].min()
        elif operation == "mean": # Corrected from ""mean"
            result = df[
                plan["column"]
            ].mean()
        elif operation == "groupby_sum":
            result = (
                df.groupby(
                    plan["group_column"]
                )[
                    plan["value_column"]
                ].sum().sort_values(ascending=False))
        elif operation == "groupby_mean":
            result = (df.groupby(
                    plan["group_column"])["value_column"].mean()) # Ensure 'value_column' is correctly used here
        elif operation == "sort":
            result = (df.sort_values(by=plan["column"],ascending=False))
        elif operation == "top_n":
            result = (df.nlargest(plan["n"],plan["column"]))
    except Exception as e:
        result = str(e)
    state["query_result"] = result
    return state

# Explanation Agent (Copied from MLyXnLhM7OeM)
def answer_agent(state):
    prompt = f"""
You are a Senior Data Analyst.
User Question
{state["user_question"]}
Actual Query Result
{state["query_result"]}
Explain the result professionally.
Do not modify the numbers.
Only explain.
"""
    response = llm.invoke(prompt)
    state["answer"] = response.content
    return state

# Analysis Graph
analysis_graph = StateGraph(AgentState)
analysis_graph.add_node("profiling",profiling_agent)
analysis_graph.add_node("planner",tool_planner_agent)
analysis_graph.add_node("executor",robust_tool_executor)
analysis_graph.add_node("insight",insight_agent)
analysis_graph.set_entry_point("profiling")
analysis_graph.add_edge("profiling","planner")
analysis_graph.add_edge("planner","executor")
analysis_graph.add_edge("executor","insight")
analysis_graph.add_edge("insight",END)
analysis_workflow = analysis_graph.compile()

# Chat Graph
chat_graph = StateGraph(AgentState)
chat_graph.add_node("planner",query_planner_agent)
chat_graph.add_node("executor",pandas_query_engine)
chat_graph.add_node("answer",answer_agent)
chat_graph.add_node("memory",save_memory)
chat_graph.set_entry_point("planner")
chat_graph.add_edge("planner","executor")
chat_graph.add_edge("executor","answer")
chat_graph.add_edge("answer","memory")
chat_graph.add_edge("memory",END)
chat_workflow = chat_graph.compile()

# Report Graph
report_graph = StateGraph(AgentState)
report_graph.add_node("insight",insight_agent)
report_graph.add_node("memory",memory_agent)
report_graph.set_entry_point("insight")
report_graph.add_edge("insight","memory")
report_graph.add_edge("memory",END)
report_workflow = report_graph.compile()

# Master Agent
class UniversalAIAgent:
    def __init__(self):
        self.analysis = analysis_workflow
        self.chat = chat_workflow
        self.report = report_workflow
    def analyze(self, state):
        return self.analysis.invoke(state)
    def chat_with_data(self, state):
        return self.chat.invoke(state)
    def generate_report(self, state):
        return self.report.invoke(state)

# Create Agent
agent = UniversalAIAgent()

In [25]:
# Analyze Dataset
analysis_result = agent.analyze({
    "dataframe":df,
    "profile":{},
    "charts":[],
    "query_plan":{},
    "query_result":"",
    "insight":"",
    "answer":"",
    "errors":[],
    "memory":[],
    "user_question":"Analyze this dataset."})

# Ask Question
chat_result = agent.chat_with_data({
    "dataframe":df,
    "profile":profile,
    "query_plan":{},
    "query_result":"",
    "user_question":
    "Which product has highest sales?",
    "answer":"",
    "memory":[]})

# Generate Report
report = agent.generate_report({
    "dataframe":df,
    "profile":profile,
    "charts":[],
    "insight":analysis_result["insight"],
    "memory":memory.get(),
    "user_question":"Generate final report."
})

JSON Decode Error in tool_planner_agent: Expecting value: line 1 column 1 (char 0)
LLM Response: ```json
{
  "tools": [
    {
      "name": "histogram",
      "params": {
        "column": "Sales_Amount"
      }
    },
    {
      "name": "bar_chart",
      "params": {
        "category": "Region",
        "value": "Sales_Amount"
      }
    },
    {
      "name": "bar_chart",
      "params": {
        "category": "Product_Category",
        "value": "Sales_Amount"
      }
    },
    {
      "name": "heatmap",
      "params": {}
    },
    {
      "name": "scatter",
      "params": {
        "column1": "Unit_Cost",
        "column2": "Unit_Price"
      }
    }
  ]
}
```
